<a href="https://colab.research.google.com/github/honeybadger-REX/english-german-transformer/blob/main/ENCODER-DECODER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TRANSFORMER ENGLISH -> GERMAN
# WITH ENCODER PREDICTION STAGE
# ============================================================

# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

!pip install -q datasets sacrebleu


# ============================================================
# 2. IMPORTS
# ============================================================

import math
import random
import torch
import sacrebleu

from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset


# ============================================================
# 3. RANDOM SEED
# ============================================================

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 5. LOAD MULTI30K DATASET
# ============================================================

dataset = load_dataset("bentrevett/multi30k")

print(dataset)

print("\nExample:")
print("English:", dataset["train"][0]["en"])
print("German :", dataset["train"][0]["de"])


# ============================================================
# 6. CONFIGURATION
# ============================================================

MAX_LENGTH = 60

# Start with 10,000 examples.
# Once everything works, increase this.
TRAIN_SIZE = 20000

BATCH_SIZE = 32

D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 512
DROPOUT = 0.1

LEARNING_RATE = 3e-4

NUM_EPOCHS = 20

# Weight of encoder prediction loss
ENCODER_LOSS_WEIGHT = 0.3

# Percentage of source tokens masked during training
MASK_PROBABILITY = 0.15


# ============================================================
# 7. SIMPLE TOKENIZER
# ============================================================

class STI:

    def __init__(self):

        self.token2idx = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<CLI>": 2,   # beginning token
            "<SEP>": 3,   # ending token
            "<MAK>": 4
        }

        self.idx2token = {
            0: "<PAD>",
            1: "<UNK>",
            2: "<CLI>",
            3: "<SEP>",
            4: "<MAK>"
        }


    def vocab(self, sentences):

        for sentence in sentences:

            words = sentence.lower().split()

            for word in words:

                if word not in self.token2idx:

                    idx = len(self.token2idx)

                    self.token2idx[word] = idx
                    self.idx2token[idx] = word

        return self.token2idx, self.idx2token


    def tokenizer(self, sentence):

        tokens = sentence.lower().split()

        tokens = (
            ["<CLI>"]
            + tokens
            + ["<SEP>"]
        )

        ids = []

        for word in tokens:

            if word in self.token2idx:
                ids.append(self.token2idx[word])

            else:
                ids.append(
                    self.token2idx["<UNK>"]
                )


        # Truncate
        ids = ids[:MAX_LENGTH]


        # Padding
        while len(ids) < MAX_LENGTH:

            ids.append(
                self.token2idx["<PAD>"]
            )


        return torch.tensor(
            ids,
            dtype=torch.long
        )


# ============================================================
# 8. BUILD ENGLISH AND GERMAN VOCABULARIES
# ============================================================

english_tokenizer = STI()
german_tokenizer = STI()


english_tokenizer.vocab(
    dataset["train"][:TRAIN_SIZE]["en"]
)

german_tokenizer.vocab(
    dataset["train"][:TRAIN_SIZE]["de"]
)


SRC_VOCAB_SIZE = len(
    english_tokenizer.token2idx
)

TGT_VOCAB_SIZE = len(
    german_tokenizer.token2idx
)


print("\nEnglish vocabulary:", SRC_VOCAB_SIZE)
print("German vocabulary :", TGT_VOCAB_SIZE)


# ============================================================
# 9. SPECIAL TOKEN IDS
# ============================================================

PAD_ID = english_tokenizer.token2idx["<PAD>"]
BOS_ID = english_tokenizer.token2idx["<CLI>"]
EOS_ID = english_tokenizer.token2idx["<SEP>"]

GERMAN_BOS_ID = german_tokenizer.token2idx["<CLI>"]
GERMAN_EOS_ID = german_tokenizer.token2idx["<SEP>"]
GERMAN_PAD_ID = german_tokenizer.token2idx["<PAD>"]


# ============================================================
# 10. DATASET CLASS
# ============================================================

class TranslationDataset(Dataset):

    def __init__(
        self,
        dataset,
        english_tokenizer,
        german_tokenizer
    ):

        self.data = dataset

        self.english_tokenizer = english_tokenizer
        self.german_tokenizer = german_tokenizer


    def __len__(self):

        return len(self.data)


    def __getitem__(self, idx):

        english = self.data[idx]["en"]
        german = self.data[idx]["de"]

        source = self.english_tokenizer.tokenizer(
            english
        )

        target = self.german_tokenizer.tokenizer(
            german
        )

        return source, target


# ============================================================
# 11. CREATE TRAIN DATASET
# ============================================================

train_hf = dataset["train"].select(
    range(TRAIN_SIZE)
)

train_data = TranslationDataset(
    train_hf,
    english_tokenizer,
    german_tokenizer
)


# ============================================================
# 12. DATA LOADER
# ============================================================

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)


print(
    "\nNumber of training examples:",
    len(train_data)
)

source, target = train_data[0]

print("\nExample tokenized source:")
print(source)

print("\nExample tokenized target:")
print(target)


# ============================================================
# 13. POSITIONAL + TOKEN EMBEDDING
# ============================================================

class PositionEmbedding(nn.Module):

    def __init__(
        self,
        max_length,
        d_model,
        vocab_size
    ):

        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            max_length,
            d_model
        )


    def forward(self, x):

        B, S = x.shape
        print(S)
        positions = torch.arange(
            S,
            device=x.device
        ).unsqueeze(0)

        token_emb = self.token_embedding(x)

        position_emb = self.position_embedding(
            positions
        )

        return token_emb + position_emb


# ============================================================
# 14. MULTI-HEAD ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        dropout=0.1
    ):

        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads

        self.head_dim = d_model // num_heads

        self.wq = nn.Linear(
            d_model,
            d_model
        )

        self.wk = nn.Linear(
            d_model,
            d_model
        )

        self.wv = nn.Linear(
            d_model,
            d_model
        )

        self.wo = nn.Linear(
            d_model,
            d_model
        )

        self.dropout = nn.Dropout(dropout)

        self.scale = math.sqrt(
            self.head_dim
        )


    def split_heads(self, x):

        B, S, _ = x.shape

        x = x.view(
            B,
            S,
            self.num_heads,
            self.head_dim
        )

        x = x.transpose(1, 2)

        return x


    def forward(
        self,
        q,
        k,
        v,
        mask=None
    ):

        B, S_q, _ = q.shape

        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        )

        scores = scores / self.scale

        if mask is not None:

            scores = scores.masked_fill(
                mask == 0,
                float("-inf")
            )

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        context = torch.matmul(
            attention_weights,
            v
        )

        context = (
            context
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                S_q,
                self.d_model
            )
        )

        return self.wo(context)


# ============================================================
# 15. FEED FORWARD NETWORK
# ============================================================

class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout=0.1
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            d_model,
            d_ff
        )

        self.fc2 = nn.Linear(
            d_ff,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(self, x):

        x = self.fc1(x)

        x = torch.relu(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x


# ============================================================
# 16. ENCODER LAYER
# ============================================================

class EncoderLayer(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads,
            dropout
        )

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.drop1 = nn.Dropout(
            dropout
        )

        self.drop2 = nn.Dropout(
            dropout
        )


    def forward(self, x):

        residual = x

        x = self.norm1(x)

        x = self.self_attn(
            x,
            x,
            x
        )

        x = residual + self.drop1(x)


        residual = x

        x = self.norm2(x)

        x = self.ffn(x)

        x = residual + self.drop2(x)

        return x


# ============================================================
# 17. TRANSFORMER ENCODER
# ============================================================

class TransformerEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        num_layers,
        d_ff,
        max_length,
        dropout=0.1
    ):

        super().__init__()

        self.embedding = PositionEmbedding(
            max_length,
            d_model,
            vocab_size
        )

        self.layers = nn.ModuleList([

            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout
            )

            for _ in range(num_layers)

        ])

        self.final_norm = nn.LayerNorm(
            d_model
        )


    def forward(self, src):

        x = self.embedding(src)

        for layer in self.layers:

            x = layer(x)

        x = self.final_norm(x)

        return x


# ============================================================
# 18. DECODER LAYER
# ============================================================

class DecoderLayer(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads,
            dropout
        )

        self.cross_attn = MultiHeadAttention(
            d_model,
            num_heads,
            dropout
        )

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.norm3 = nn.LayerNorm(
            d_model
        )

        self.drop1 = nn.Dropout(
            dropout
        )

        self.drop2 = nn.Dropout(
            dropout
        )

        self.drop3 = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        encoder_output,
        causal_mask
    ):

        # --------------------------------
        # Masked self attention
        # --------------------------------

        residual = x

        x_norm = self.norm1(x)

        x_attn = self.self_attn(
            x_norm,
            x_norm,
            x_norm,
            causal_mask
        )

        x = residual + self.drop1(
            x_attn
        )


        # --------------------------------
        # Cross attention
        # --------------------------------

        residual = x

        x_norm = self.norm2(x)

        x_attn = self.cross_attn(
            x_norm,
            encoder_output,
            encoder_output
        )

        x = residual + self.drop2(
            x_attn
        )


        # --------------------------------
        # Feed forward
        # --------------------------------

        residual = x

        x_norm = self.norm3(x)

        x_ffn = self.ffn(x_norm)

        x = residual + self.drop3(
            x_ffn
        )

        return x


# ============================================================
# 19. TRANSFORMER DECODER
# ============================================================

class TransformerDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        num_layers,
        d_ff,
        max_length,
        dropout=0.1
    ):

        super().__init__()

        self.embedding = PositionEmbedding(
            max_length,
            d_model,
            vocab_size
        )

        self.layers = nn.ModuleList([

            DecoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout
            )

            for _ in range(num_layers)

        ])

        self.final_norm = nn.LayerNorm(
            d_model
        )

        self.predictor = nn.Linear(
            d_model,
            vocab_size
        )


    def forward(
        self,
        target,
        encoder_output,
        causal_mask
    ):

        x = self.embedding(target)

        for layer in self.layers:

            x = layer(
                x,
                encoder_output,
                causal_mask
            )

        x = self.final_norm(x)

        logits = self.predictor(x)

        return logits


# ============================================================
# 20. CAUSAL MASK
# ============================================================

def create_causal_mask(
    seq_len,
    device
):

    return torch.tril(
        torch.ones(
            seq_len,
            seq_len,
            device=device
        )
    )


# ============================================================
# 21. MASK SOURCE TOKENS FOR ENCODER PREDICTION
# ============================================================

def create_masked_source(
    source,
    mask_probability=0.15
):

    masked_source = source.clone()

    prediction_mask = (
        torch.rand(
            source.shape,
            device=source.device
        )
        < mask_probability
    )

    # Do not mask special/padding tokens
    prediction_mask &= (
        source != PAD_ID
    )

    prediction_mask &= (
        source != BOS_ID
    )

    prediction_mask &= (
        source != EOS_ID
    )

    # Replace selected tokens with <UNK>
    masked_source[prediction_mask] = (
        english_tokenizer.token2idx["<UNK>"]
    )

    return masked_source


# ============================================================
# 22. COMPLETE TRANSFORMER
#
# English
#    ↓
# Encoder
#    ↓
# English prediction
#    ↓
# Soft predicted English representation
#    ↓
# German decoder
# ============================================================

class Transformer(nn.Module):

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model,
        num_heads,
        num_layers,
        d_ff,
        max_length,
        dropout=0.1
    ):

        super().__init__()


        # --------------------------------
        # Encoder
        # --------------------------------

        self.encoder = TransformerEncoder(
            vocab_size=src_vocab_size,
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers,
            d_ff=d_ff,
            max_length=max_length,
            dropout=dropout
        )


        # --------------------------------
        # Encoder prediction head
        # --------------------------------

        self.encoder_predictor = nn.Linear(
            d_model,
            src_vocab_size
        )


        # --------------------------------
        # Convert predicted English
        # probability distribution
        # back into embeddings
        # --------------------------------

        self.predicted_embedding = nn.Embedding(
            src_vocab_size,
            d_model
        )


        # --------------------------------
        # Decoder
        # --------------------------------

        self.decoder = TransformerDecoder(
            vocab_size=tgt_vocab_size,
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers,
            d_ff=d_ff,
            max_length=max_length,
            dropout=dropout
        )


    def encode_and_predict(
        self,
        source
    ):

        # Encoder representation
        encoder_hidden = self.encoder(
            source
        )


        # Predict English vocabulary
        encoder_prediction = (
            self.encoder_predictor(
                encoder_hidden
            )
        )


        # Convert logits into probability
        prediction_probs = torch.softmax(
            encoder_prediction,
            dim=-1
        )


        # Soft predicted English embedding
        predicted_representation = torch.matmul(
            prediction_probs,
            self.predicted_embedding.weight
        )


        return (
            encoder_hidden,
            encoder_prediction,
            predicted_representation
        )


    def forward(
        self,
        source,
        target,
        causal_mask
    ):

        (
            encoder_hidden,
            encoder_prediction,
            predicted_representation
        ) = self.encode_and_predict(
            source
        )


        # German decoder receives
        # predicted English representation
        logits = self.decoder(
            target,
            predicted_representation,
            causal_mask
        )


        return (
            logits,
            encoder_prediction
        )


# ============================================================
# 23. CREATE MODEL
# ============================================================

model = Transformer(

    src_vocab_size=SRC_VOCAB_SIZE,

    tgt_vocab_size=TGT_VOCAB_SIZE,

    d_model=D_MODEL,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS,

    d_ff=D_FF,

    max_length=MAX_LENGTH,

    dropout=DROPOUT

).to(device)


print("\nModel created.")

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


# ============================================================
# 24. LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=0
)


# ============================================================
# 25. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 26. TRAINING
# ============================================================

print("\nStarting training...\n")


for epoch in range(NUM_EPOCHS):

    model.train()

    total_loss = 0.0
    total_translation_loss = 0.0
    total_encoder_loss = 0.0


    for batch_idx, (
        source,
        target
    ) in enumerate(train_loader):

        source = source.to(device)

        target = target.to(device)


        # ========================================
        # Encoder prediction training input
        # ========================================

        masked_source = create_masked_source(
            source,
            MASK_PROBABILITY
        )


        # ========================================
        # Decoder teacher forcing
        # ========================================

        decoder_input = target[:, :-1]

        decoder_target = target[:, 1:]


        # ========================================
        # Causal mask
        # ========================================

        causal_mask = create_causal_mask(
            decoder_input.size(1),
            device
        )


        # ========================================
        # Forward
        # ========================================

        logits, encoder_prediction = model(
            masked_source,
            decoder_input,
            causal_mask
        )


        # ========================================
        # German translation loss
        # ========================================

        translation_loss = criterion(

            logits.reshape(
                -1,
                logits.size(-1)
            ),

            decoder_target.reshape(
                -1
            )

        )


        # ========================================
        # English encoder prediction loss
        # ========================================

        encoder_loss = criterion(

            encoder_prediction.reshape(
                -1,
                encoder_prediction.size(-1)
            ),

            source.reshape(
                -1
            )

        )


        # ========================================
        # Combined loss
        # ========================================

        loss = (
            translation_loss
            +
            ENCODER_LOSS_WEIGHT
            * encoder_loss
        )


        # ========================================
        # Backpropagation
        # ========================================

        optimizer.zero_grad()

        loss.backward()


        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        # ========================================
        # Statistics
        # ========================================

        total_loss += loss.item()

        total_translation_loss += (
            translation_loss.item()
        )

        total_encoder_loss += (
            encoder_loss.item()
        )


    avg_loss = (
        total_loss
        /
        len(train_loader)
    )

    avg_translation_loss = (
        total_translation_loss
        /
        len(train_loader)
    )

    avg_encoder_loss = (
        total_encoder_loss
        /
        len(train_loader)
    )


    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Total Loss: {avg_loss:.4f} | "
        f"Translation Loss: {avg_translation_loss:.4f} | "
        f"Encoder Loss: {avg_encoder_loss:.4f}"
    )


# ============================================================
# 27. ENGLISH ENCODER PREDICTION
#
# This uses greedy/argmax prediction for evaluation.
# Do NOT use top-k/top-p sampling when calculating BLEU.
# ============================================================

def predict_english(
    sentence,
    max_length=MAX_LENGTH
):

    model.eval()


    source = english_tokenizer.tokenizer(
        sentence
    ).unsqueeze(0).to(device)


    with torch.no_grad():

        encoder_hidden = model.encoder(
            source
        )

        encoder_logits = (
            model.encoder_predictor(
                encoder_hidden
            )
        )


        # Greedy prediction
        predicted_ids = torch.argmax(
            encoder_logits,
            dim=-1
        )[0]


    tokens = []


    for idx in predicted_ids.tolist():

        if idx == PAD_ID:
            continue

        if idx == BOS_ID:
            continue

        if idx == EOS_ID:
            break

        token = english_tokenizer.idx2token[
            idx
        ]

        tokens.append(token)


    return " ".join(tokens)


# ============================================================
# 28. ENGLISH PREDICTION WITH TOP-K / TOP-P / TEMPERATURE
#
# This is for demonstrating sampling.
# Do NOT use this version for BLEU.
# ============================================================

def predict_english_sampled(
    sentence,
    temperature=1.0,
    top_k=50,
    top_p=0.9
):

    model.eval()


    source = english_tokenizer.tokenizer(
        sentence
    ).unsqueeze(0).to(device)


    with torch.no_grad():

        encoder_hidden = model.encoder(
            source
        )

        logits = model.encoder_predictor(
            encoder_hidden
        )


        # Temperature
        logits = logits / temperature


        # Top-k
        if top_k is not None:

            k = min(
                top_k,
                logits.size(-1)
            )

            top_k_values, _ = torch.topk(
                logits,
                k,
                dim=-1
            )

            min_top_k = (
                top_k_values[:, :, -1]
                .unsqueeze(-1)
            )

            logits = torch.where(
                logits < min_top_k,
                torch.full_like(
                    logits,
                    float("-inf")
                ),
                logits
            )


        # Top-p
        if top_p is not None:

            sorted_logits, sorted_indices = (
                torch.sort(
                    logits,
                    descending=True,
                    dim=-1
                )
            )

            sorted_probs = torch.softmax(
                sorted_logits,
                dim=-1
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1
            )

            remove_mask = (
                cumulative_probs > top_p
            )

            remove_mask[:, :, 1:] = (
                remove_mask[:, :, :-1]
                .clone()
            )

            remove_mask[:, :, 0] = False

            sorted_logits = (
                sorted_logits.masked_fill(
                    remove_mask,
                    float("-inf")
                )
            )

            logits = torch.full_like(
                logits,
                float("-inf")
            )

            logits.scatter_(
                -1,
                sorted_indices,
                sorted_logits
            )


        probs = torch.softmax(
            logits,
            dim=-1
        )


        predicted_ids = torch.multinomial(
            probs.reshape(
                -1,
                probs.size(-1)
            ),
            num_samples=1
        ).reshape(
            probs.size(0),
            probs.size(1)
        )[0]


    tokens = []


    for idx in predicted_ids.tolist():

        if idx == PAD_ID:
            continue

        if idx == BOS_ID:
            continue

        if idx == EOS_ID:
            break

        tokens.append(
            english_tokenizer.idx2token[idx]
        )


    return " ".join(tokens)


# ============================================================
# 29. GERMAN TRANSLATION
#
# English
#   ↓
# Encoder
#   ↓
# English prediction
#   ↓
# Soft English representation
#   ↓
# German Decoder
#   ↓
# German tokens
# ============================================================

def translate(
    sentence,
    max_length=MAX_LENGTH
):

    model.eval()


    source = english_tokenizer.tokenizer(
        sentence
    ).unsqueeze(0).to(device)


    generated = [
        GERMAN_BOS_ID
    ]


    with torch.no_grad():

        # ----------------------------------------
        # Encoder prediction stage
        # ----------------------------------------

        (
            encoder_hidden,
            encoder_prediction,
            predicted_representation
        ) = model.encode_and_predict(
            source
        )


        # ----------------------------------------
        # Autoregressive German generation
        # ----------------------------------------

        for _ in range(max_length - 1):

            decoder_input = torch.tensor(
                generated,
                dtype=torch.long,
                device=device
            ).unsqueeze(0)


            causal_mask = create_causal_mask(
                decoder_input.size(1),
                device
            )


            logits = model.decoder(
                decoder_input,
                predicted_representation,
                causal_mask
            )


            next_token_logits = (
                logits[:, -1, :]
            )


            next_token = torch.argmax(
                next_token_logits,
                dim=-1
            ).item()


            generated.append(
                next_token
            )


            if next_token == GERMAN_EOS_ID:

                break


    # ----------------------------------------
    # Convert IDs → German words
    # ----------------------------------------

    tokens = []


    for idx in generated:

        if idx == GERMAN_BOS_ID:
            continue

        if idx == GERMAN_EOS_ID:
            break

        if idx == GERMAN_PAD_ID:
            continue

        tokens.append(
            german_tokenizer.idx2token[idx]
        )


    return " ".join(tokens)


# ============================================================
# 30. TEST ONE EXAMPLE
# ============================================================

test_sentence = (
    "Two young, White males are outside near many bushes."
)


print("\n========================================")
print("SINGLE EXAMPLE")
print("========================================")

print("\nEnglish:")
print(test_sentence)


print("\nEncoder prediction:")
print(
    predict_english(
        test_sentence
    )
)


print("\nEncoder prediction with sampling:")
print(
    predict_english_sampled(
        test_sentence,
        temperature=0.8,
        top_k=50,
        top_p=0.9
    )
)


print("\nGerman translation:")
print(
    translate(
        test_sentence
    )
)


print("\nActual German:")
print(
    dataset["train"][0]["de"]
)


# ============================================================
# 31. ENCODER PREDICTION BLEU
# ============================================================

def evaluate_encoder_bleu(
    dataset_split,
    num_samples=None
):

    model.eval()


    if num_samples is None:

        num_samples = len(
            dataset_split
        )


    predictions = []
    references = []


    for i in range(
        min(
            num_samples,
            len(dataset_split)
        )
    ):

        english = dataset_split[i]["en"]


        prediction = predict_english(
            english
        )


        predictions.append(
            prediction
        )

        references.append(
            english
        )


    bleu = sacrebleu.corpus_bleu(

        predictions,

        [references],

        lowercase=True

    )


    return bleu.score


# ============================================================
# 32. TRANSLATION BLEU
# ============================================================

def evaluate_translation_bleu(
    dataset_split,
    num_samples=None
):

    model.eval()


    if num_samples is None:

        num_samples = len(
            dataset_split
        )


    predictions = []
    references = []


    for i in range(
        min(
            num_samples,
            len(dataset_split)
        )
    ):

        english = dataset_split[i]["en"]

        german = dataset_split[i]["de"]


        prediction = translate(
            english
        )


        predictions.append(
            prediction
        )

        references.append(
            german
        )


    bleu = sacrebleu.corpus_bleu(

        predictions,

        [references],

        lowercase=True

    )


    return bleu.score


# ============================================================
# 33. RUN ENCODER BLEU
# ============================================================

print("\n========================================")
print("ENCODER PREDICTION EVALUATION")
print("========================================")

encoder_bleu = evaluate_encoder_bleu(
    dataset["test"]
)

print(
    f"\nEncoder Prediction BLEU: "
    f"{encoder_bleu:.2f}"
)


# ============================================================
# 34. RUN TRANSLATION BLEU
# ============================================================

print("\n========================================")
print("TRANSLATION EVALUATION")
print("========================================")

translation_bleu = evaluate_translation_bleu(
    dataset["test"]
)

print(
    f"\nEnglish -> German BLEU: "
    f"{translation_bleu:.2f}"
)


# ============================================================
# 35. TEST MULTIPLE SENTENCES
# ============================================================

print("\n========================================")
print("TEST TRANSLATIONS")
print("========================================")


test_examples = [
    "A man is riding a bicycle.",
    "A woman is walking outside.",
    "Two dogs are playing in the park.",
    "A group of people are sitting at a table.",
    "A young boy is playing with a ball."
]


for sentence in test_examples:

    print("\n----------------------------------------")

    print("English:")
    print(sentence)

    print("\nPredicted English:")
    print(
        predict_english(
            sentence
        )
    )

    print("\nPredicted German:")
    print(
        translate(
            sentence
        )
    )


# ============================================================
# 36. SAVE MODEL
# ============================================================

torch.save(
    {
        "model_state_dict": model.state_dict(),

        "english_token2idx":
            english_tokenizer.token2idx,

        "english_idx2token":
            english_tokenizer.idx2token,

        "german_token2idx":
            german_tokenizer.token2idx,

        "german_idx2token":
            german_tokenizer.idx2token,

        "config": {
            "max_length": MAX_LENGTH,
            "d_model": D_MODEL,
            "num_heads": NUM_HEADS,
            "num_layers": NUM_LAYERS,
            "d_ff": D_FF
        }
    },
    "transformer_multi30k_encoder_predictor.pt"
)


print(
    "\nModel saved as:"
)

print(
    "transformer_multi30k_encoder_predictor.pt"
)


# ============================================================
# 37. FINAL RESULTS
# ============================================================

print("\n")
print("============================================")
print("FINAL RESULTS")
print("============================================")

print(
    f"Encoder Prediction BLEU : "
    f"{encoder_bleu:.2f}"
)

print(
    f"Translation BLEU        : "
    f"{translation_bleu:.2f}"
)

print("============================================")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 14.9 MB/s eta 0:00:00
Device: cuda
GPU: Tesla T4


README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

Example:
English: Two young, White males are outside near many bushes.
German : Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.

English vocabulary: 11666
German vocabulary : 18387

Number of training examples: 20000

Example tokenized source:
tensor([ 2,  5,  6,  7,  8,  9, 10, 11, 12, 13,  3,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0])

Example tokenized target:
tensor([ 2,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,  3,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
      